# Visual Recognition HW1 - ResNet18 & ResNet50
Full training pipeline for image classification.
Dataset structure:
data/train, data/val, data/test


In [9]:
# Imports
import os
os.environ["KMP_DUPLICATE_LIB_OK"] = "TRUE"  # workaround for OpenMP duplicate runtime on Windows
os.environ["OMP_NUM_THREADS"] = "1"          # reduce OpenMP threading issues in notebooks

import torch
import torch.nn as nn
import torch.optim as optim
from torchvision import datasets, transforms, models
from torch.utils.data import DataLoader
from tqdm import tqdm
import matplotlib
matplotlib.use("Agg")  # safer non-interactive backend for notebooks/kernels
import matplotlib.pyplot as plt

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Device:", device)

if device.type == "cuda":
    print("GPU:", torch.cuda.get_device_name(0))


Device: cuda
GPU: NVIDIA GeForce RTX 5060 Laptop GPU


In [10]:
# Data transforms
train_transform = transforms.Compose([
    transforms.RandomResizedCrop(224, scale=(0.8, 1.0), ratio=(3. / 4., 4. / 3.)),
    transforms.RandomHorizontalFlip(0.5),
    transforms.RandomVerticalFlip(0.2),
    transforms.RandomRotation(15),
    transforms.RandomAffine(degrees=10, translate=(0.05, 0.05), scale=(0.9, 1.1), shear=5),
    transforms.RandomApply([
        transforms.ColorJitter(brightness=0.15, contrast=0.15, saturation=0.15, hue=0.05)
    ], p=0.6),
    transforms.AutoAugment(transforms.AutoAugmentPolicy.IMAGENET),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
    transforms.RandomErasing(p=0.2, scale=(0.02, 0.2), ratio=(0.3, 3.3), value='random')
])

val_transform = transforms.Compose([
    transforms.Resize(256),
    transforms.CenterCrop(224),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
])

In [11]:
# Load dataset
train_dataset = datasets.ImageFolder("data/train", transform=train_transform)
val_dataset = datasets.ImageFolder("data/val", transform=val_transform)

num_classes = len(train_dataset.classes)
print("Classes:", num_classes)
print("Class names:", train_dataset.classes)

batch_size = 32
num_workers = 8  
use_pin_memory = (device.type == "cuda")

train_loader = DataLoader(
    train_dataset,
    batch_size=batch_size,
    shuffle=True,
    num_workers=num_workers,
    pin_memory=use_pin_memory,
    persistent_workers=False
)

val_loader = DataLoader(
    val_dataset,
    batch_size=batch_size,
    shuffle=False,
    num_workers=num_workers,
    pin_memory=use_pin_memory,
    persistent_workers=False
)

print("Train samples:", len(train_dataset))
print("Val samples:", len(val_dataset))
print("Train batches per epoch:", len(train_loader))
print("Val batches per epoch:", len(val_loader))
print("num_workers:", num_workers, "| pin_memory:", use_pin_memory)


Classes: 100
Class names: ['0', '1', '10', '11', '12', '13', '14', '15', '16', '17', '18', '19', '2', '20', '21', '22', '23', '24', '25', '26', '27', '28', '29', '3', '30', '31', '32', '33', '34', '35', '36', '37', '38', '39', '4', '40', '41', '42', '43', '44', '45', '46', '47', '48', '49', '5', '50', '51', '52', '53', '54', '55', '56', '57', '58', '59', '6', '60', '61', '62', '63', '64', '65', '66', '67', '68', '69', '7', '70', '71', '72', '73', '74', '75', '76', '77', '78', '79', '8', '80', '81', '82', '83', '84', '85', '86', '87', '88', '89', '9', '90', '91', '92', '93', '94', '95', '96', '97', '98', '99']
Train samples: 20724
Val samples: 300
Train batches per epoch: 648
Val batches per epoch: 10
num_workers: 8 | pin_memory: True


In [12]:
# Model builder (fast & accurate)
def get_model(name="resnet50", pretrained=True, freeze_backbone=False):
    if name == "resnet18":
        weights = models.ResNet18_Weights.IMAGENET1K_V1 if pretrained else None
        model = models.resnet18(weights=weights)
    elif name == "resnet50":
        weights = models.ResNet50_Weights.IMAGENET1K_V2 if pretrained else None
        model = models.resnet50(weights=weights)
    else:
        raise ValueError("Model name must be 'resnet18' or 'resnet50'.")

    if freeze_backbone:
        for param in model.parameters():
            param.requires_grad = False

    model.fc = nn.Sequential(
    nn.Dropout(0.5),
    nn.Linear(model.fc.in_features, num_classes)
)

    return model.to(device)

# Use resnet18 by default for faster epochs; switch to resnet50 if you need max capacity
model = get_model('resnet50', pretrained=True, freeze_backbone=False)
print(model)


ResNet(
  (conv1): Conv2d(3, 64, kernel_size=(7, 7), stride=(2, 2), padding=(3, 3), bias=False)
  (bn1): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
  (relu): ReLU(inplace=True)
  (maxpool): MaxPool2d(kernel_size=3, stride=2, padding=1, dilation=1, ceil_mode=False)
  (layer1): Sequential(
    (0): Bottleneck(
      (conv1): Conv2d(64, 64, kernel_size=(1, 1), stride=(1, 1), bias=False)
      (bn1): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
      (conv2): Conv2d(64, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
      (bn2): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
      (conv3): Conv2d(64, 256, kernel_size=(1, 1), stride=(1, 1), bias=False)
      (bn3): BatchNorm2d(256, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
      (relu): ReLU(inplace=True)
      (downsample): Sequential(
        (0): Conv2d(64, 256, kernel_size=(1, 1), stride=(1, 

In [13]:
# Training setup
# Increase epochs and use adaptive scheduling for smoother LR decay. 
epochs = 25
criterion = nn.CrossEntropyLoss(label_smoothing=0.1)

backbone_params = []
fc_params = []
for name, param in model.named_parameters():
    if name.startswith("fc"):
        fc_params.append(param)
    else:
        backbone_params.append(param)

optimizer = optim.AdamW(
    [
        {"params": fc_params, "lr": 4e-4},
        {"params": backbone_params, "lr": 8e-5}
    ],
    weight_decay=1e-2,
    eps=1e-8
)

scheduler = optim.lr_scheduler.ReduceLROnPlateau(
    optimizer,
    mode='min',
    factor=0.5,
    patience=2,
    min_lr=1e-7,
)

scaler = torch.amp.GradScaler(enabled=(device.type == "cuda"))

if device.type == "cuda":
    torch.backends.cudnn.benchmark = True

In [14]:
# Improved training loop with validation, mixed precision, and best model saving
patience = 5
counter = 0
best_val_acc = 0.0
history = {
    "train_loss": [],
    "train_acc": [],
    "val_loss": [],
    "val_acc": []
}

for epoch in range(epochs):
    # =========================
    # Train
    # =========================
    model.train()
    train_loss = 0.0
    train_correct = 0
    train_total = 0

    train_bar = tqdm(train_loader, desc=f"Epoch {epoch+1}/{epochs} [Train]", leave=False)

    for images, labels in train_bar:
        images = images.to(device, non_blocking=True)
        labels = labels.to(device, non_blocking=True)

        optimizer.zero_grad()

        with torch.amp.autocast(device_type='cuda', enabled=device.type == 'cuda'):
            outputs = model(images)
            loss = criterion(outputs, labels)

        scaler.scale(loss).backward()
        scaler.step(optimizer)
        scaler.update()

        train_loss += loss.item()
        _, preds = outputs.max(1)
        train_correct += preds.eq(labels).sum().item()
        train_total += labels.size(0)

        train_bar.set_postfix({
            "loss": f"{loss.item():.4f}",
            "acc": f"{train_correct / train_total:.4f}"
        })

    avg_train_loss = train_loss / len(train_loader)
    train_acc = train_correct / train_total

    # =========================
    # Validation
    # =========================
    model.eval()
    val_loss = 0.0
    val_correct = 0
    val_total = 0

    with torch.no_grad():
        val_bar = tqdm(val_loader, desc=f"Epoch {epoch+1}/{epochs} [Val]", leave=False)

        for images, labels in val_bar:
            images = images.to(device, non_blocking=True)
            labels = labels.to(device, non_blocking=True)

            with torch.amp.autocast(device_type='cuda', enabled=device.type == 'cuda'):
                outputs = model(images)
                loss = criterion(outputs, labels)

            val_loss += loss.item()
            _, preds = outputs.max(1)
            val_correct += preds.eq(labels).sum().item()
            val_total += labels.size(0)

            val_bar.set_postfix({
                "loss": f"{loss.item():.4f}",
                "acc": f"{val_correct / val_total:.4f}"
            })

    avg_val_loss = val_loss / len(val_loader)
    val_acc = val_correct / val_total

    # Save history
    history["train_loss"].append(avg_train_loss)
    history["train_acc"].append(train_acc)
    history["val_loss"].append(avg_val_loss)
    history["val_acc"].append(val_acc)

    # Save best model
    if val_acc > best_val_acc:
        best_val_acc = val_acc
        torch.save(model.state_dict(), "best_model.pth")
        best_msg = " <-- best model saved"
        counter = 0
    else:
        best_msg = ""
        counter += 1

    # Scheduler from validation loss
    scheduler.step(avg_val_loss)

    print(
        f"Epoch {epoch+1}/{epochs} | "
        f"Train Loss: {avg_train_loss:.4f} | Train Acc: {train_acc:.4f} | "
        f"Val Loss: {avg_val_loss:.4f} | Val Acc: {val_acc:.4f}"
        f"{best_msg}"
    )

    if counter >= patience:
        print("Early stopping triggered")
        break

print(f"\nTraining finished. Best validation accuracy: {best_val_acc:.4f}")

Epoch 1/25 | Train Loss: 2.6557 | Train Acc: 0.4719 | Val Loss: 1.9746 | Val Acc: 0.6333 <-- best model saved


Epoch 2/25 | Train Loss: 1.8111 | Train Acc: 0.6969 | Val Loss: 1.6448 | Val Acc: 0.7367 <-- best model saved


Epoch 3/25 | Train Loss: 1.6172 | Train Acc: 0.7558 | Val Loss: 1.4863 | Val Acc: 0.7900 <-- best model saved


Epoch 4/25 | Train Loss: 1.5047 | Train Acc: 0.7892 | Val Loss: 1.4535 | Val Acc: 0.7767


Epoch 5/25 | Train Loss: 1.4441 | Train Acc: 0.8059 | Val Loss: 1.4146 | Val Acc: 0.8333 <-- best model saved


...

Epoch 21/25 | Train Loss: 1.0210 | Train Acc: 0.9384 | Val Loss: 1.2183 | Val Acc: 0.8767


Epoch 22/25 | Train Loss: 1.0181 | Train Acc: 0.9399 | Val Loss: 1.2210 | Val Acc: 0.8833 <-- best model saved


Epoch 23/25 | Train Loss: 1.0016 | Train Acc: 0.9443 | Val Loss: 1.2506 | Val Acc: 0.8733


Epoch 24/25 | Train Loss: 1.0047 | Train Acc: 0.9443 | Val Loss: 1.2451 | Val Acc: 0.8667


Epoch 25/25 | Train Loss: 0.9944 | Train Acc: 0.9462 | Val Loss: 1.2441 | Val Acc: 0.8800

Training finished. Best validation accuracy: 0.8833


In [17]:
import os
import csv
from PIL import Image
from torch.utils.data import Dataset, DataLoader

# =========================
# Test transform
# =========================
test_transform = transforms.Compose([
    transforms.Resize(256),
    transforms.CenterCrop(224),
    transforms.ToTensor(),
    transforms.Normalize(
        mean=[0.485, 0.456, 0.406],
        std=[0.229, 0.224, 0.225]
    )
])

# =========================
# Custom test dataset
# =========================
class TestDataset(Dataset):
    def __init__(self, image_dir, transform=None):
        self.image_dir = image_dir
        self.transform = transform
        self.image_names = sorted([
            fname for fname in os.listdir(image_dir)
            if fname.lower().endswith((".png", ".jpg", ".jpeg", ".bmp", ".webp"))
        ])

    def __len__(self):
        return len(self.image_names)

    def __getitem__(self, idx):
        image_name = self.image_names[idx]
        image_path = os.path.join(self.image_dir, image_name)

        image = Image.open(image_path).convert("RGB")

        if self.transform:
            image = self.transform(image)

        return image, image_name

# =========================
# Load best model
# =========================
model = get_model(name="resnet50", pretrained=False, freeze_backbone=False)
model.load_state_dict(torch.load("first.pth", map_location=device))
model.eval()

# =========================
# Build test loader
# =========================
test_dataset = TestDataset("data/test", transform=test_transform)
print("Test image count:", len(test_dataset))
if len(test_dataset) == 0:
    raise RuntimeError("No images found in data/test. Check path and image extensions.")

test_loader = DataLoader(
    test_dataset,
    batch_size=32,
    shuffle=False,
    num_workers=0,
    pin_memory=(device.type == "cuda")
)

# =========================
# Predict
# =========================
predictions = []
processed = 0

with torch.no_grad():
    for images, image_names in test_loader:
        images = images.to(device, non_blocking=True)

        with torch.amp.autocast(device_type='cuda', enabled=device.type == 'cuda'):
            outputs = model(images)
            preds = outputs.argmax(dim=1)

        preds = preds.cpu().numpy()

        for image_name, pred_idx in zip(image_names, preds):
            base_name = os.path.splitext(image_name)[0]
            pred_class = train_dataset.classes[pred_idx]
            predictions.append([base_name, pred_class])

        processed += len(image_names)
        print(f"Predicted {processed}/{len(test_dataset)}", end="\r")

print(f"\nPrediction complete: {len(predictions)} images")

# =========================
# Save CSV
# =========================
csv_path = "prediction.csv"

with open(csv_path, mode="w", newline="", encoding="utf-8") as f:
    writer = csv.writer(f)
    writer.writerow(["image_name", "pred_label"])
    writer.writerows(predictions)

print(f"CSV saved to: {csv_path}")

Test image count: 2344
Predicted 2344/2344
Prediction complete: 2344 images
CSV saved to: prediction.csv


In [16]:
# Safe plotting (convert values to floats and save figure instead of interactive display)
from math import isfinite

def _to_float_list(values, name):
    cleaned = []
    for i, x in enumerate(values):
        if hasattr(x, "detach"):
            x = x.detach()
        if hasattr(x, "cpu"):
            x = x.cpu()
        if hasattr(x, "item"):
            x = x.item()
        x = float(x)
        if not isfinite(x):
            raise ValueError(f"{name}[{i}] is not finite: {x}")
        cleaned.append(x)
    return cleaned

train_loss = _to_float_list(history["train_loss"], "train_loss")
val_loss = _to_float_list(history["val_loss"], "val_loss")
train_acc = _to_float_list(history["train_acc"], "train_acc")
val_acc = _to_float_list(history["val_acc"], "val_acc")

fig, axes = plt.subplots(1, 2, figsize=(10, 4))

axes[0].plot(train_loss, label="train_loss", marker="o")
axes[0].plot(val_loss, label="val_loss", marker="o")
axes[0].set_xlabel("Epoch")
axes[0].set_ylabel("Loss")
axes[0].legend()
axes[0].grid(True)

axes[1].plot(train_acc, label="train_acc", marker="o")
axes[1].plot(val_acc, label="val_acc", marker="o")
axes[1].set_xlabel("Epoch")
axes[1].set_ylabel("Accuracy")
axes[1].legend()
axes[1].grid(True)

fig.tight_layout()
plot_path = "training_curves.png"
fig.savefig(plot_path, dpi=150, bbox_inches="tight")
plt.close(fig)

print(f"Plot saved to: {os.path.abspath(plot_path)}")


Plot saved to: c:\Users\titou\Documents\ISEN\M1\Taiwan\Cours\Selected Topic In Visual Recognition Using Deep Learning\HW1\training_curves.png
